In [ ]:
# 1. 시스템 패키지 업데이트 및 Java(Mecab 등 텍스트 처리에 필요) 설치
!apt-get update -y
!apt-get install -y openjdk-8-jdk

# 2. 파이썬 필수 패키지 설치
!pip install --upgrade pip
!pip install konlpy gensim soynlp soyspacing bokeh networkx lxml sentencepiece

# 3. Mecab (형태소 분석기) 설치
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)
!pip install mecab-python3

# 4. FastText 설치
!pip install fasttext

# 5. ratsgo 님의 embedding 저장소 클론 (데이터셋/코드용)
!git clone https://github.com/ratsgo/embedding.git

한국어 위키백과의 raw data를 다운로드

In [ ]:
!wget https://dumps.wikimedia.org/kowiki/latest/kowiki-latest-pages-articles.xml.bz2 -P /notebooks/embedding/data/raw

In [ ]:
#raw data에서 특수문자, 공백, 이메일 주소 등 불필요한 문자열을 제거하여 순수 text만 추출하도록 하는 함수
import re
from gensim.utils import to_unicode

#삭제할 문자
WIKI_REMOVE_CHARS = re.compile("'+|(=+.{2,30}=+)|__TOC__|(ファイル:).+|:(en|de|it|fr|es|kr|zh|no|fi):|\n", re.UNICODE)
WIKI_SPACE_CHARS = re.compile("(\\s|゙|゚|　)+", re.UNICODE)
EMAIL_PATTERN = re.compile("(^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$)", re.UNICODE)
URL_PATTERN = re.compile("(ftp|http|https)?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+", re.UNICODE)
WIKI_REMOVE_TOKEN_CHARS = re.compile("(\\*$|:$|^파일:.+|^;)", re.UNICODE)
MULTIPLE_SPACES = re.compile(' +', re.UNICODE)

def tokenize(content,token_min_len=2,token_max_len=100,lower=True):
  content=re.sub(EMAIL_PATTERN,' ',content)
  content=re.sub(URL_PATTERN,' ',content)
  content=re.sub(WIKI_REMOVE_CHARS,' ',content)
  content=re.sub(WIKI_SPACE_CHARS,' ',content)
  content=re.sub(MULTIPLE_SPACES,' ',content)
  tokens=content.replace(",)","").split(" ")
  result=[]
  for token in tokens:
    if not token.startswith('_'):
      token_candidate=to_unicode(re.sub(WIKI_REMOVE_TOKEN_CHARS,'',token))

    else:
      token_candidate=""
    if len(token_candidate)>0:
      result.append(token_candidate)
  return result

<>:8: SyntaxWarning: invalid escape sequence '\.'
<>:9: SyntaxWarning: invalid escape sequence '\('
<>:8: SyntaxWarning: invalid escape sequence '\.'
<>:9: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_4355/907701466.py:8: SyntaxWarning: invalid escape sequence '\.'
  EMAIL_PATTERN = re.compile("(^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$)", re.UNICODE)
/tmp/ipykernel_4355/907701466.py:9: SyntaxWarning: invalid escape sequence '\('
  URL_PATTERN = re.compile("(ftp|http|https)?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+", re.UNICODE)


In [ ]:
#위에서 다운 받은 raw data를 전처리한다.
from gensim.corpora import WikiCorpus
from gensim.utils import to_unicode

in_f="/notebooks/embedding/data/raw/kowiki-latest-pages-articles.xml.bz2" #raw data 경로
out_f="/notebooks/embedding/data/processed_wiki_ko.txt" #output data 경로
output=open(out_f,'w+')
wiki=WikiCorpus(in_f,tokenizer_func=tokenize)
i=0
for text in wiki.get_texts(): #method get_texts()을 이용하여 raw data를 text로 변환한다.
  output.write(bytes(' '.join(text),'utf-8').decode('utf-8')+'\n')
  i+=1
  if(i%10000==0):
    print('Processed' + str(i)+'articles')

output.close()
print('Processing complete!')


한국어 기계 독해를 위한 dataset KorQuAD 다운로드

In [ ]:
!wget https://korquad.github.io/dataset/KorQuAD_v1.0_train.json -P /notebooks/embedding/data/raw
!wget https://korquad.github.io/dataset/KorQuAD_v1.0_dev.json -P /notebooks/embedding/data/raw

In [ ]:
#KorQuAD 전처리
import json

corpus_fname="/notebooks/embedding/data/raw/KorQuAD_v1.0_train.json"
output_fname="/notebooks/embedding/data/processed_korquad_train.txt"

with open(corpus_fname) as f1, open(output_fname,'w',encoding='utf-8') as f2:
  dataset_json=json.load(f1)
  dataset=dataset_json['data']
  for article in dataset:
    w_lines=[]
    for paragraph in article['paragraphs']:
      w_lines.append(paragraph['context'])
      for qa in paragraph['gas']:
        q_text=qa['question']
        for a in qa['answer']:
          a_text=a['text']
          w_lines.append(q_text+" "+a_text)
    for line in w_lines:
      f2.writelines(line+'\n')


KeyError: 'gas'

네이버 영화 리뷰 말뭉치 다운로드

In [ ]:
!wget https://github.com/e9t/nsmc/raw/master/ratings.txt -P /notebooks/embedding/data/raw
!wget https://github.com/e9t/nsmc/raw/master/ratings_train.txt -P /notebooks/embedding/data/raw
!wget https://github.com/e9t/nsmc/raw/master/ratings_test.txt -P /notebooks/embedding/data/raw

In [ ]:
#전처리
corpus_path="/notebooks/embedding/data/raw/ratings.txt"
output_fname="/notebooks/embedding/data/processed_ratings.txt"
with_label=False    #label과 함께 저장하고 싶으면 True로 설정, 리뷰들만 저장하고 싶으면 False로 설정한다.

with open(corpus_path,'r',encoding='utf-8') as f1, open(output_fname,'w',encoding='utf-8') as f2:
  next(f1)  #skip head line
  for line in f1:
    _, sentence, label=line.strip().split('\t')
    if not sentence: continue
    if with_label:
      f2.writelines(sentence + "\u241E"+label+"\n")
    else:
      f2.writelines(sentence+"\n")

tokenizer(형태소 분석기)--지도 학습 기반

In [ ]:
#은전한닢
from konlpy.tag import Mecab
tokenizer=Mecab()
tokenizer.morphs("아버지가방에들어가신다")

In [ ]:
#품사 정보 확인
tokenizer.pos("아버지가방에들어가신다")

In [ ]:
#tokenizer 선택 함수
from konlpy.tag import Okt,Komoran,Mecab,Hannanum,Kkma

def get_tokenizer(tokenizer_name):
  if tokenizer_name=="komoran":
    tokenizer=Komoran()
  elif tokenizer_name=="okt":
    tokenizer=Okt()
  elif tokenizer_name=="mecab":
    tokenizer=Mecab()
  elif tokenizer_name=="hannanum":
    tokenizer=Hannanum()
  elif tokenizer_name=="kkma":
    tokenizer=Kkma()
  else:
    tokenizer=Mecab()
  return tokenizer

#코모란이라는 tokenizer 사용 예시
tokenizer=get_tokenizer("komoran")
tokenizer.morphs("아버지가방에들어가신다")
tokenizer.pos("아버지가방에들어가신다")

tokenizer Khaiii

In [ ]:
!pip install khaii

In [ ]:
from khaii import KhaiiiApi
tokenizer=KhaiiiApi()

In [ ]:
data=tokenizer.analyze("아버지가방에들어가신다")
tokens=[]
for word in data:
  tokens.extend([str(m).split("/")[0] for m in word.morphs])

In [ ]:
#품사 정보 확인
data=tokenizer.analyze("아버지가방에들어가신다")
tokens=[]
for word in data:
  tokens.extend([str(m) for m in word.morphs])

중요 tokens 처리 방법

In [ ]:
from konlpy.tag import Mecab
tokenizer=Mecab()
tokenizer.morphs("가우스전자 텔레비전 정말 좋네요")

tokenizer(형태소 분석기)--비지도 학습 기반

In [ ]:
#tokenizer soynlp
#soynlp 단어 점수 학습
from soynlp.word import WordExtractor

corpus_fname="/notebooks/embedding/data/processed/processed_ratings.txt"
model_fname="/notebooks/embedding/data/processed/soyword.model"
sentences=[sent.strip() for sent in open(corpus_fname,'r').readlines()]
word_extractor=WordExtractor(min_frequency=100,
                             min_cohesion_forward=0.05,
                             min_right_branching_entropy=0.0)
word_extractor.train(sentences)
word_extractor.save(model_fname)

In [ ]:
#soynlp 형태소 분석
import math
from soynlp.word import WordExtractor
from soynlp.tokenizer import LTokenizer

model_fname="/notebooks/embedding/data/processed/soyword.model"

word_extractor=WordExtractor(min_frequency=100,
                             min_cohesion_forward=0.05,
                             main_right_branching_entropy=0.0)

word_extractor.load(model_fname)
scores=word_extractor.word_scores()
scores={key:(scores[key].cohesion_forward*math.exp(scores[key].right_branchin_entropy)) for key in scores.keys()}
tokenizer=LTokenizer(scores=scores)
tokens=tokenizer.tokenize("애비는 종이었다")

In [ ]:
#Byte Pair Encoding을 이용하여 vocabulary을 만든다.
import sentencepiece as spm
train="""--input=/notebooks/embedding/data/processed/processed_Wiki_ko.txt  \
         --model_prefix=sentpiece \
         --vocab_size=32000 \
         --model_type=bpe --character_coverage=0.9995"""

spm.SentencePieceTrainer.Train(train)

띄어쓰기 교정

In [ ]:
from soyspacing.countbase import CountSpace

corpus_fname="/notebooks/embedding/data/processed/processed_ratings.txt"
model_fname="/notebooks/embedding/data/processed/space-correct.model"

#soynlp 띄어쓰기 교정 모듈 학습
model=CountSpace()
model.train(corpus_fname)
model.save_model(model_fname,json_format=False)

#예시
model.correct("어릴때보고 지금다시봐도 재밌어요")